# Interfaze 1.1 Lite — modèle composite (OCR, détection, traduction)

Interfaze regroupe plusieurs modèles derrière une seule API (`model.ocr(...)`). Score annoncé : 83,8 sur olmOCR-Bench,
rien de publié sur le malgache. **Il ne tourne pas sur un poste ordinaire** : lancez ce notebook sur une machine
avec un GPU de 80 Go (H100, A100 80 Go, location cloud). `trust_remote_code=True` exécute le code du dépôt HF :
relisez-le avant.

| | |
|---|---|
| Modèle | [interfaze-ai/interfaze-1-lite](https://huggingface.co/interfaze-ai/interfaze-1-lite) |
| Licence | interfaze-open (licence propre, à lire avant tout usage) |
| Taille | modèle composite (PP-OCRv5 pour les lignes + dots.mocr pour la lecture) |
| Matériel | 1 GPU de 80 Go, compute capability ≥ 8.9 (testé sur H100) ; `ffmpeg` installé |
| Sortie | texte + pages → lignes → mots, avec boîtes et confiance |
| PDF | pages rastérisées avec `pypdfium2` (fonction commune `load_document`) |

Aucune clé d'API ni token HuggingFace n'est nécessaire : les poids publics sont téléchargés au premier lancement
(dans `~/.cache/huggingface`). Le notebook se lance depuis le dossier `hf_ocr_notebooks/` (il importe `ocr_common.py`).

In [ ]:
!hf download interfaze-ai/interfaze-1-lite requirements.txt --local-dir interfaze_req
%pip install -q -r interfaze_req/requirements.txt
%pip install -q pypdfium2 pillow pandas rapidfuzz datasets matplotlib

## 1. Jeux de données

Toutes les sources écrivent dans `data/<source>/` une image par échantillon et, à côté, sa vérité terrain `.txt` :

| Source | Contenu |
|---|---|
| `synthetic/` | 6 images + 1 PDF (français, anglais, tableau, petite police, rotation, bruit) |
| `malagasy/` | 5 images de phrases malgaches intégrées (sans réseau) |
| `hf_Lo-Renz-O_malagasy-sentence/` | phrases malgaches réelles du Hub, rendues en images (bruit / inclinaison) |
| `hf_Teklia_IAM-line/` | vraies images de lignes manuscrites anglaises (IAM) |
| `custom/` | vos documents (image ou PDF, + `nom.txt` optionnel pour mesurer le CER) |

Les datasets du Hub sont téléchargés une seule fois (paquet `datasets`) ; sans réseau, ils sont ignorés.

In [ ]:
import time
from pathlib import Path

from IPython.display import display

import ocr_common as oc

oc.make_test_set()
oc.make_malagasy_set()
for loader in (oc.load_hf_text_dataset, oc.load_hf_image_dataset):
    try:
        loader()
    except Exception as e:  # pas de réseau / pas de paquet datasets
        print(f"{loader.__name__} ignoré : {type(e).__name__}: {e}")

FILES = oc.list_inputs()
print(f"{len(FILES)} fichiers :", ", ".join(f"{f.parent.name}/{f.name}" for f in FILES))
oc.show_dataset([f for f in FILES if f.suffix != ".pdf"])

In [ ]:
import tempfile

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    vram = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"GPU : {torch.cuda.get_device_name(0)}, {vram:.0f} Go")
    if vram < 75:
        print("⚠ moins de 80 Go de VRAM : le chargement risque d'échouer")
else:
    print("⚠ pas de GPU : Interfaze ne peut pas tourner ici")
print("Matériel :", oc.hardware(), "| DEVICE =", DEVICE)

## 2. Chargement du modèle

In [ ]:
t0 = time.perf_counter()
from transformers import AutoModel

model = AutoModel.from_pretrained("interfaze-ai/interfaze-1-lite", trust_remote_code=True)
LOAD_S = time.perf_counter() - t0
print(f"Modèle chargé en {LOAD_S:.1f}s sur {DEVICE}")

## 3. Fonction d'extraction


`model.ocr` prend un chemin de fichier : chaque page est écrite dans un PNG temporaire, donc les coordonnées des
lignes sont en pixels de la page.

In [ ]:
TMP = Path(tempfile.mkdtemp())


def predict(image):
    path = TMP / "page.png"
    image.save(path)
    doc = model.ocr(str(path))
    boxes = [[*line["bounds"]["top_left"], *line["bounds"]["bottom_right"]]
             for section in doc["sections"] for line in section["lines"]]
    return doc["text"], boxes


def extract(image):
    return predict(image)[0]

## 4. Validation visuelle

Pour chaque image : à gauche la page avec les zones détectées (en rouge), à droite la vérité terrain et le texte lu,
avec le CER dans le titre. Commencez par une image malgache, puis parcourez tout le jeu de test.

In [ ]:
img = oc.load_document(oc.MALAGASY_DIR / "mg_vaovao.png")[0]
t = time.perf_counter()
text, boxes = predict(img)
print(f"({time.perf_counter() - t:.1f}s)\n{text}")
oc.show_result(img, text, boxes, ref=oc.ground_truth(oc.MALAGASY_DIR / "mg_vaovao.png"), title="mg_vaovao.png")

In [ ]:
# toutes les sources sauf les PDF ; augmentez max_files pour tout voir
oc.visual_check(predict, [f for f in FILES if f.suffix != ".pdf"], max_files=12)

### PDF multi-pages

In [ ]:
pdf_path = oc.SYNTH_DIR / "rapport_2_pages.pdf"
for i, page in enumerate(oc.load_document(pdf_path), 1):
    t = time.perf_counter()
    text = extract(page)
    print(f"--- page {i} ({time.perf_counter() - t:.1f}s)\n{text}\n")

## 5. Benchmark sur tout le jeu de test

Temps par page et précision (CER / WER) sur tous les fichiers. Le premier appel inclut un éventuel temps de compilation/« chauffe ».
Les textes extraits sont enregistrés dans `results/<modèle>/` et les mesures dans `results/<modèle>.csv`
(utilisées par le notebook `00_comparaison.ipynb`).

In [ ]:
MODEL_NAME = "Interfaze-1.1-Lite"
df = oc.run_benchmark(extract, MODEL_NAME, FILES)
oc.save_meta(MODEL_NAME, repo="interfaze-ai/interfaze-1-lite", load_s=LOAD_S, device=str(DEVICE))
display(oc.summarize(df).to_frame(MODEL_NAME))

In [ ]:
display(df[[c for c in ("file", "type", "pages", "s_per_page", "cer", "wer", "error") if c in df]].round(3))
mg = df[df.file.str.startswith("mg_") | df.file.str.fullmatch(r"\d{3}\.png")]
if "cer" in mg and len(mg):
    print(f"CER moyen sur les échantillons malgaches et HF : {mg.cer.mean():.1%}")
oc.show_comparison(df, n=3)

## 6. Utiliser le modèle sur vos propres fichiers

Remplacez le chemin ci-dessous par une image ou un PDF ; le texte est enregistré à côté du fichier (`.ocr.txt`).

In [ ]:
def extract_file(path, dpi=200, max_pages=None):
    path = Path(path)
    pages = oc.load_document(path, dpi=dpi, max_pages=max_pages)
    text = "\n\n".join(extract(p) for p in pages)
    path.with_suffix(".ocr.txt").write_text(text, encoding="utf-8")
    return text


# print(extract_file("data/custom/mon_document.pdf"))